Collect Cloud coverage for various zip codes

In [1]:
# import library
import ee
import os
import geemap
import geopandas as gpd
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from tqdm import tqdm


In [2]:
# authenticate & initilialize earth engine
ee.Authenticate()
ee.Initialize(project='ee-jfelix-netmetering')

In [3]:
# set working directory/mount drive
from google.colab import drive
drive.mount('/content/drive')

#  Set working directory inside Google Drive
drive_path = '/content/drive/My Drive/net_metering'
os.makedirs(drive_path, exist_ok=True)
os.chdir(drive_path)

print("Current working directory:", os.getcwd())

# to keep console from disconnecting ctr+shift+i to inspector and go to the console
# function ClickConnect(){
#    console.log("Working");
#    document.querySelector("colab-toolbar-button#connect").click()
#}
#setInterval(ClickConnect,60000)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working directory: /content/drive/My Drive/net_metering


In [ ]:
# use dataset = ee.ImageCollection('NOAA/NCEP_DOE_RE2/total_cloud_coverage')
# band = "tcdc" (total cloud coverage in percentage: 0 to 100)
# filter dates :2011 to 2020

In [4]:
# get the boundary for all the offices
# you have to mount the google drive
# load the shapefile for all the utility boundaries (This was simplified in R)
all_sf = gpd.read_file('shapefiles/utility_boundaries_simplified_annual_v2.shp')


/usr/local/lib/python3.11/dist-packages/pyogrio/raw.py:198: RuntimeWarning: shapefiles/utility_boundaries_simplified_annual_v2.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


In [5]:
# view the boundary shapefile
all_sf.head()
all_sf.tail()

,year,full_id,full_id_y,geometry
5148,2012.0,UT_11135,UT_11135_2012,"POLYGON ((-111.99107 41.83314, -111.64558 41.7..."
5149,2013.0,UT_11135,UT_11135_2013,"POLYGON ((-111.99107 41.83314, -111.64558 41.7..."
5150,2011.0,VA_19882,VA_19882_2011,"POLYGON ((-80.61584 37.22951, -80.35152 37.349..."
5151,2012.0,VA_19882,VA_19882_2012,"POLYGON ((-80.61584 37.22951, -80.35152 37.349..."
5152,2013.0,VA_19882,VA_19882_2013,"POLYGON ((-80.61584 37.22951, -80.35152 37.349..."


In [ ]:

# This is the information for the google earth engine data that has cloud coverage
# https://developers.google.com/earth-engine/datasets/catalog/NOAA_NCEP_DOE_RE2_total_cloud_coverage

# description: NCEP-DOE Reanalysis 2 project is using a state-of-the-art analysis/forecast system to perform data assimilation using past data from 1979 through the previous year.

# citation: NCEP-DOE AMIP-II Reanalysis (R-2): M. Kanamitsu, W. Ebisuzaki, J. Woollen, S-K Yang, J.J. Hnilo, M. Fiorino, and G. L. Potter. 1631-1643, Nov 2002, Bulletin of the American Meteorological Society..

# availability: 1979 to 2025

# provider: NOAA

# Cadence: 6 hrs
# Pixel size: 278300 meters
# bands: tcdc , units: %, min: 0*, max: 100*, description: Total Cloud Cover
# ee snippet: ee.ImageCollection("NOAA/NCEP_DOE_RE2/total_cloud_coverage")


In [6]:
# bring the data from ee and select the band
tcc_collection = ee.ImageCollection('NOAA/NCEP_DOE_RE2/total_cloud_coverage').select('tcdc')

In [7]:
# List of unique utility IDs
ulist = all_sf['full_id_y'].unique().tolist()

In [8]:
# create an empty df to collect data for loop
utility_monthly_tcc = []

In [ ]:
# Loop over each utility
for u in tqdm(ulist):  # u = "AZ_24211_2011"
    start_time = datetime.now()

    u_sf = all_sf[all_sf['full_id_y'] == u]

    # Add an explicit check for None geometry before attempting to access its attributes
    if u_sf.empty or u_sf.geometry.is_empty.any() or u_sf.geometry.values[0] is None:
        print(f"Skipping utility {u} due to empty or invalid geometry.")
        continue

    # Convert geometry to EE
    geom_json = u_sf.geometry.values[0].__geo_interface__
    boundary = ee.Geometry(geom_json)

    # Extract year
    year = int(u_sf['year'].values[0])
    d1 = datetime(year, 1, 1)
    d2 = d1 + relativedelta(months=12)

    # Filter image collection
    monthly_image = tcc_collection.filterDate(d1.strftime('%Y-%m-%d'), d2.strftime('%Y-%m-%d'))

    # Function to extract average TCC for each image
    def extract_tcc(image):
        date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd HH:mm')

        med_tcc = image.reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=boundary,
            scale=5000,
            maxPixels=1e13
        ).get('tcdc')

        mean_tcc = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=boundary,
            scale=5000,
            maxPixels=1e13
        ).get('tcdc')
        return ee.Feature(None, {'date': date, 'avg_tcc': mean_tcc, 'median_tcc': med_tcc})

    # Map over collection and get features
    tcc_features = monthly_image.map(extract_tcc)
    tcc_table = ee.FeatureCollection(tcc_features)

    # Export to client
    try:
        tcc_dicts = geemap.ee_to_geojson(tcc_table)
        tcc_df = pd.DataFrame(tcc_dicts['features'])
        tcc_df['date'] = pd.to_datetime(tcc_df['properties'].apply(lambda x: x['date']))
        tcc_df['avg_tcc'] = tcc_df['properties'].apply(lambda x: x['avg_tcc'])
        tcc_df['median_tcc'] = tcc_df['properties'].apply(lambda x: x['median_tcc'])
        tcc_df.drop(columns=['properties', 'type', 'geometry'], inplace=True)

        # create the hour and month and year for tcc_df
        tcc_df['hour'] = tcc_df['date'].dt.hour
        tcc_df['day'] = tcc_df['date'].dt.day
        tcc_df['month'] = tcc_df['date'].dt.month
        tcc_df['year'] = tcc_df['date'].dt.year

        # filter out midnitght, so that we only count from 6 am to 6 pm
        tcc_df = tcc_df[tcc_df['hour'] != 0 ]

        # aggregate cloud coverage at the daily level
        tcc_df = tcc_df.groupby(['year', 'month', 'day']).agg({'avg_tcc': 'mean', 'median_tcc': 'median'}).reset_index()

        # count the number of cloudy days by those with coverage over 75% on average
        tcc_df['cloudy_days'] = tcc_df['avg_tcc'] > 75

        # now aggreagate to the monthly level, number of cloudy days, avg_tcc and median_tcc
        tcc_df = tcc_df.groupby(['year', 'month']).agg({'cloudy_days': 'sum', 'avg_tcc': 'mean', 'median_tcc': 'median'}).reset_index()

        # Add utility info
        for col in u_sf.columns:
            if col != 'geometry':
                tcc_df[col] = u_sf.iloc[0][col]

        utility_monthly_tcc.append(tcc_df)

    except Exception as e:
        print(f"Failed for utility {u}: {e}")
        continue

    print(f"{u} completed in {datetime.now() - start_time}")

# Combine all data and write to CSV
result_df = pd.concat(utility_monthly_tcc, ignore_index=True)

# Save to CSV
result_df.to_csv("data files/monthly_cloudy_days_utility_all_75pcnt_cov.csv", index=False)


  0%|          | 1/4292 [00:15<17:54:05, 15.02s/it]

AK_1651_2011 completed in 0:00:15.018733


  0%|          | 2/4292 [00:26<15:30:46, 13.02s/it]

AK_7353_2011 completed in 0:00:11.616342


  0%|          | 3/4292 [00:37<14:19:31, 12.02s/it]

...

IN_17633_2022 completed in 0:00:04.865218


 73%|███████▎  | 3137/4292 [6:52:19<2:06:58,  6.60s/it]

KS_5860_2022 completed in 0:00:04.766572


 73%|███████▎  | 3138/4292 [6:52:23<1:55:17,  5.99s/it]

KS_9996_2022 completed in 0:00:04.589145


In [ ]:
result_df.head()

,year,month,cloudy_days,avg_tcc,median_tcc,eiaid,utility,full_id,full_id_y
0,2011.0,1,1,27.707476,5.199951,19728.0,"UNS Electric, Inc",AZ_19728,AZ_19728_2011
1,2011.0,2,3,33.373883,10.649902,19728.0,"UNS Electric, Inc",AZ_19728,AZ_19728_2011
2,2011.0,3,4,36.572689,28.399902,19728.0,"UNS Electric, Inc",AZ_19728,AZ_19728_2011
3,2011.0,4,1,31.474212,23.149902,19728.0,"UNS Electric, Inc",AZ_19728,AZ_19728_2011
4,2011.0,5,0,23.497922,21.199951,19728.0,"UNS Electric, Inc",AZ_19728,AZ_19728_2011


In [2]:
utility_monthly_tcc.tail(30)

NameError: name 'utility_monthly_tcc' is not defined

[link text](https://)keep typing something: 1011